<a href="https://colab.research.google.com/github/sj-workbench/years_of_learnings/blob/main/binning%26binarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer

In [27]:
df = pd.read_csv("/content/Titanic-Dataset.csv", usecols = ['Age','Fare','Survived'])
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [28]:
df.isnull().sum()

,0
Survived,0
Age,177
Fare,0


In [29]:
df.dropna(inplace = True)

In [30]:
df.isnull().sum()

,0
Survived,0
Age,0
Fare,0


In [31]:
x = df.drop('Survived', axis = 1)
y = df['Survived']

In [32]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)
print(x_train.shape)
print(x_test.shape)

(571, 2)
(143, 2)


In [33]:
clf = DecisionTreeClassifier()
clf.fit(x_train, y_train)
y_pred = clf.predict(x_test)
accuracy_score(y_test, y_pred)

0.6293706293706294

In [34]:
np.mean(cross_val_score(DecisionTreeClassifier(), x, y, cv = 10, scoring = 'accuracy'))

np.float64(0.6246870109546165)

In [35]:
kbin_age = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')
kbin_fare = KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')

In [36]:
trf = ColumnTransformer([
    ('first',kbin_age,[0]),
    ('second',kbin_fare,[1])
])

In [37]:
x_train_trf = trf.fit_transform(x_train)
x_test_trf = trf.transform(x_test)

In [38]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.42,  6.  , 16.  , 19.  , 21.  , 23.  , 25.  , 28.  , 30.  ,
              32.  , 35.  , 38.  , 42.  , 47.  , 54.  , 80.  ])             ],
      dtype=object)

In [39]:
trf.named_transformers_['second'].bin_edges_

array([array([  0.    ,   7.25  ,   7.775 ,   7.8958,   8.1583,  10.5   ,
               13.    ,  14.4542,  18.75  ,  26.    ,  26.55  ,  31.275 ,
               51.4792,  76.2917, 108.9   , 512.3292])                   ],
      dtype=object)

In [40]:
output = pd.DataFrame({
    'age':x_train['Age'],
    'age_trf':x_train_trf[:,0],
    'fare':x_train['Fare'],
    'fare_trf':x_train_trf[:,1]
})

In [41]:
output['age_labels'] = pd.cut(x=x_train['Age'],
                                    bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=x_train['Fare'],
                                    bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [42]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
60,22.0,4.0,7.2292,0.0,"(21.0, 23.0]","(0.0, 7.25]"
21,34.0,9.0,13.0000,6.0,"(32.0, 35.0]","(10.5, 13.0]"
587,60.0,14.0,79.2000,13.0,"(54.0, 80.0]","(76.292, 108.9]"
605,36.0,10.0,15.5500,7.0,"(35.0, 38.0]","(14.454, 18.75]"
575,19.0,3.0,14.5000,7.0,"(16.0, 19.0]","(14.454, 18.75]"


In [43]:
clf = DecisionTreeClassifier()
clf.fit(x_train_trf,y_train)
y_pred2 = clf.predict(x_test_trf)

In [44]:
accuracy_score(y_test,y_pred2)

0.6363636363636364

In [45]:
x_trf = trf.fit_transform(x)
np.mean(cross_val_score(DecisionTreeClassifier(),x,y,cv=10,scoring='accuracy'))

np.float64(0.6233176838810641)